# Annotation — Manual Relevant-Span Labelling

Interactive tool for marking which spans of a conversation snippet carry the user-specific evidence, used to sanity-check the relevance of the supplied snippets.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip -q install -U pandas pyarrow ipywidgets
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )

SUBSET_NAME = "all_subset"
SUBSETS_DIR = PP_ROOT / "v2_personamem_persona_subsets"
ANNOT_DIR = PP_ROOT / "annotation" / "relevant_span_labels"
ANNOT_DIR.mkdir(parents=True, exist_ok=True)

print("PP_ROOT:", PP_ROOT.resolve())
print("Annotations dir:", ANNOT_DIR.resolve())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 70.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
Mounted at /content/drive
PP_ROOT: /content/drive/MyDrive/privacy_perserving_pllm
Annotations dir: /content/drive/MyDrive/privacy_perserving_pllm/annotation/relevant_span_labels


In [2]:
import ast
import html
import json
import random
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import pandas as pd

SEED = 42
N_SAMPLES = 50
RESUME = True
ANNOTATOR_ID = "asma"

LABELS_PATH = ANNOT_DIR / f"{SUBSET_NAME}_val_relevant_spans.jsonl"
SAMPLE_INDEX_PATH = ANNOT_DIR / f"{SUBSET_NAME}_val_sample_ids_seed{SEED}_n{N_SAMPLES}.json"

random.seed(SEED)


def load_subset_val(name=SUBSET_NAME, subsets_dir=SUBSETS_DIR):
    subset_dir = Path(subsets_dir) / name
    if not subset_dir.exists():
        raise FileNotFoundError(f"Subset not found: {subset_dir}")
    with open(subset_dir / "personas.json", "r", encoding="utf-8") as f:
        persona_ids = [int(p) for p in json.load(f)]
    va = pd.read_parquet(subset_dir / "val.parquet")
    va = va[va["persona_id"].isin(persona_ids)].reset_index(drop=True)
    return persona_ids, va


def parse_user_query(raw):
    if isinstance(raw, dict):
        return str(raw.get("content", raw)).strip()
    if isinstance(raw, str):
        try:
            d = ast.literal_eval(raw)
            if isinstance(d, dict):
                return str(d.get("content", raw)).strip()
        except (ValueError, SyntaxError):
            pass
        return raw.strip()
    return str(raw).strip()


def get_snippet_raw(raw):
    """Same formatting as training notebooks: str(related_conversation_snippet).strip()."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return ""
    return str(raw).strip()


def split_words(text):
    """Whitespace split — same as indexing convention for this labeling task."""
    return (text or "").split()


persona_ids, val_df = load_subset_val()
val_df = val_df.copy()
val_df["row_id"] = val_df.index.astype(int)
print(f"Loaded {SUBSET_NAME} val: {len(val_df):,} rows | {len(persona_ids)} personas")
print("Has columns correct_answer / preference:",
      "correct_answer" in val_df.columns, "/", "preference" in val_df.columns)
if "preference" not in val_df.columns:
    raise KeyError(
        "Column `preference` is missing from val.parquet. "
        f"Available columns: {list(val_df.columns)}"
    )
print("Example preference:", repr(val_df["preference"].iloc[0])[:200])
print("Example correct_answer:", repr(val_df["correct_answer"].iloc[0])[:200])
_raw0 = get_snippet_raw(val_df["related_conversation_snippet"].iloc[0])
print("Snippet format preview (training/raw):", repr(_raw0[:180]))
print("n words (split):", len(split_words(_raw0)))

Loaded all_subset val: 714 rows | 150 personas
Has columns correct_answer / preference: True / True
Example preference: 'Struggled with conflict in marriage over differing long-term relocation plans.'
Example correct_answer: 'It might help to create a calm, neutral space — maybe over a pot of herbal tea — where you both can explore each other’s hopes without jumping to logistics. Just like balancing aesthetics and sustain
Snippet format preview (training/raw): '[{"role": "user", "content": "Could you please help me refine the language in this professional email so it sounds more polished and clear?\\n\\nSubject: Regarding Proposal for the S'
n words (split): 264


In [3]:
def load_existing_labels(path=LABELS_PATH):
    out: Dict[int, Dict[str, Any]] = {}
    if not path.exists():
        return out
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            out[int(rec["row_id"])] = rec
    return out

def append_label(rec, path=LABELS_PATH):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

existing = load_existing_labels()
print(f"Existing labels: {len(existing)} -> {LABELS_PATH.name}")

if SAMPLE_INDEX_PATH.exists():
    with open(SAMPLE_INDEX_PATH, "r", encoding="utf-8") as f:
        sample_ids = [int(x) for x in json.load(f)]
    print(f"Reusing sample list ({len(sample_ids)} ids)")
else:
    all_ids = val_df["row_id"].tolist()
    rng = random.Random(SEED)
    sample_ids = all_ids[:] if N_SAMPLES >= len(all_ids) else rng.sample(all_ids, N_SAMPLES)
    rng.shuffle(sample_ids)
    with open(SAMPLE_INDEX_PATH, "w", encoding="utf-8") as f:
        json.dump(sample_ids, f, indent=2)
    print(f"Created sample list ({len(sample_ids)} ids)")

todo_ids = [i for i in sample_ids if not (RESUME and i in existing)]
print(f"To label now: {len(todo_ids)} / {len(sample_ids)}")

queue: List[Dict[str, Any]] = []
for rid in todo_ids:
    row = val_df.loc[val_df["row_id"] == rid].iloc[0]
    snippet = get_snippet_raw(row.get("related_conversation_snippet"))
    correct_answer = row["correct_answer"]
    preference = row["preference"]
    if correct_answer is None or (isinstance(correct_answer, float) and pd.isna(correct_answer)):
        correct_answer = ""
    if preference is None or (isinstance(preference, float) and pd.isna(preference)):
        preference = ""
    queue.append({
        "row_id": int(rid),
        "persona_id": int(row["persona_id"]),
        "query": parse_user_query(row.get("user_query")),
        "correct_answer": str(correct_answer).strip(),
        "preference": str(preference).strip(),
        "snippet": snippet,
        "words": split_words(snippet),
        "n_words": len(split_words(snippet)),
    })

print("Queue preview (row_id, persona_id, n_words):",
      [(q["row_id"], q["persona_id"], q["n_words"]) for q in queue[:5]])
if queue:
    print("--- first example ---")
    print("preference     :", queue[0]["preference"][:300])
    print("correct_answer :", queue[0]["correct_answer"][:300])
    print("snippet[:200]  :", queue[0]["snippet"][:200])

Existing labels: 1 -> all_subset_val_relevant_spans.jsonl
Reusing sample list (50 ids)
To label now: 49 / 50
Queue preview (row_id, persona_id, n_words): [(558, 449, 391), (114, 923, 338), (300, 637, 377), (348, 661, 272), (95, 861, 478)]
--- first example ---
preference     : Likes trying new restaurants
correct_answer : Since you enjoy trying new restaurants, you might spend part of the weekend exploring Madison’s evolving food scene—perhaps sampling a few innovative farm-to-table spots or a fresh take on Scandinavian cuisine. You could pair that with a visit to a local museum or an evening performance at the Overt
snippet[:200]  : [{"role": "user", "content": "Could you help me refine the language in this section of my annual research impact report to make it read more clearly and professionally?\n\n---\nIn addition to analyzin


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output as colab_output
    HAS_COLAB_CALLBACKS = True
except Exception:
    HAS_COLAB_CALLBACKS = False

if not queue:
    print("Nothing left to label. Increase N_SAMPLES or set RESUME=False.")
else:
    state = {
        "i": 0,
        "saved": 0,
        "skipped": 0,
        "spans": [],
        "pending_start": None,
    }

    header = widgets.HTML(layout=widgets.Layout(width="100%"))
    query_out = widgets.HTML(layout=widgets.Layout(width="100%"))
    correct_out = widgets.HTML(layout=widgets.Layout(width="100%"))
    pref_out = widgets.HTML(layout=widgets.Layout(width="100%"))
    snippet_plain = widgets.HTML(layout=widgets.Layout(width="100%"))
    instruct = widgets.HTML(layout=widgets.Layout(width="100%"))
    words_html = widgets.HTML(layout=widgets.Layout(width="100%"))
    spans_out = widgets.HTML(layout=widgets.Layout(width="100%"))
    result_out = widgets.HTML(layout=widgets.Layout(width="100%"))
    status = widgets.HTML(layout=widgets.Layout(width="100%"))

    start_box = widgets.BoundedIntText(
        value=0, min=0, max=10_000, description="word_start",
        style={"description_width": "90px"}, layout=widgets.Layout(width="220px"),
    )
    end_box = widgets.BoundedIntText(
        value=0, min=0, max=10_000, description="word_end",
        style={"description_width": "90px"}, layout=widgets.Layout(width="220px"),
    )
    btn_add = widgets.Button(description="Add phrase (from boxes)", button_style="info")
    btn_undo = widgets.Button(description="Undo last phrase")
    btn_clear = widgets.Button(description="Clear phrases")
    btn_cancel_sel = widgets.Button(description="Cancel selection")
    btn_submit = widgets.Button(description="Submit all phrases", button_style="success")
    btn_skip = widgets.Button(description="Skip", button_style="warning")
    btn_back = widgets.Button(description="Back")
    btn_next = widgets.Button(description="Next example")

    def current():
        return queue[state["i"]]

    def _covered_indices():
        covered = set()
        for sp in state["spans"]:
            for j in range(sp["word_start"], sp["word_end"] + 1):
                covered.add(j)
        return covered

    def _word_button_style(i):
        covered = _covered_indices()
        pending = state["pending_start"]
        if pending is not None and i == pending:
            bg = "#64b5f6"
        elif i in covered:
            bg = "#ffd54f"
        else:
            bg = "#e6e6e6"
        return (
            "color:#000000 !important;"
            f"background:{bg} !important;"
            "border:1px solid #333;"
            "margin:2px;"
            "padding:4px 7px;"
            "cursor:pointer;"
            "font-size:13px;"
            "border-radius:4px;"
            "display:inline-block;"
        )

    def _render_clickable_words(words):
        parts = []
        for i, w in enumerate(words):
            style = _word_button_style(i)
            label = html.escape(w)
            idx = f"<span style='color:#333;font-size:10px;'>{i}</span>"
            if HAS_COLAB_CALLBACKS:
                onclick = (
                    "google.colab.kernel.invokeFunction("
                    f"'span_word_click',[{i}],{{}}) "
                )
                parts.append(
                    f"<button type='button' style=\"{style}\" onclick=\"{onclick}\">"
                    f"{idx} {label}</button>"
                )
            else:
                parts.append(
                    f"<span style=\"{style}\">{idx} {label}</span>"
                )
        note = ""
        if not HAS_COLAB_CALLBACKS:
            note = (
                "<p style='color:#000;'><i>Clickable HTML callbacks need Colab. "
                "Locally, use the word_start / word_end boxes below.</i></p>"
            )
        return (
            "<div style='max-height:420px;overflow:auto;border:1px solid #999;"
            "padding:8px;background:#fafafa;'>"
            + "".join(parts)
            + "</div>"
            + note
        )

    def _make_span(words, s, e):
        if s > e:
            s, e = e, s
        if not words:
            raise ValueError("Snippet has no words.")
        if s < 0 or e >= len(words):
            raise ValueError(
                f"Invalid indices [{s}, {e}] for {len(words)} words "
                f"(valid: 0 .. {len(words) - 1})"
            )
        return {
            "word_start": int(s),
            "word_end": int(e),
            "phrase": " ".join(words[s : e + 1]),
            "n_words": int(e - s + 1),
        }

    def _refresh_spans_html():
        spans = state["spans"]
        if not spans:
            spans_out.value = "<p style='color:#000;'><b>Added phrases:</b> <i>none yet</i></p>"
            return
        items = []
        for k, sp in enumerate(spans, start=1):
            items.append(
                f"<li style='color:#000;margin-bottom:6px;'>"
                f"<b>#{k}</b> [{sp['word_start']}, {sp['word_end']}] "
                f"({sp['n_words']} words)<br>"
                f"<pre style='white-space:pre-wrap;background:#fff8e1;padding:8px;"
                f"border:1px solid #f0c36d;color:#000;margin:4px 0;'>"
                f"{html.escape(sp['phrase'])}</pre></li>"
            )
        spans_out.value = (
            f"<p style='color:#000;'><b>Added phrases:</b> {len(spans)}</p>"
            f"<ol>{''.join(items)}</ol>"
        )

    def _set_instruct():
        if state["pending_start"] is None:
            instruct.value = (
                "<p style='color:#000;background:#e3f2fd;padding:8px;border:1px solid #90caf9;'>"
                "<b>Click the START word</b> of the next relevant phrase.</p>"
            )
        else:
            instruct.value = (
                f"<p style='color:#000;background:#fff3e0;padding:8px;border:1px solid #ffb74d;'>"
                f"<b>Start = {state['pending_start']}</b>. Now click the <b>END word</b>.</p>"
            )

    def _refresh_words():
        words_html.value = _render_clickable_words(current()["words"])

    def on_word_index(idx):
        words = current()["words"]
        idx = int(idx)
        if idx < 0 or idx >= len(words):
            return
        if state["pending_start"] is None:
            state["pending_start"] = idx
            result_out.value = (
                f"<p style='color:#000;'>Start selected: <b>{idx}</b> "
                f"({html.escape(words[idx][:100])})</p>"
            )
        else:
            s = state["pending_start"]
            e = idx
            try:
                sp = _make_span(words, s, e)
            except Exception as err:
                result_out.value = f"<p style='color:#b00020'>{html.escape(str(err))}</p>"
                state["pending_start"] = None
                _set_instruct()
                _refresh_words()
                return
            key = (sp["word_start"], sp["word_end"])
            if any((x["word_start"], x["word_end"]) == key for x in state["spans"]):
                result_out.value = (
                    f"<p style='color:#b00020'>Span [{key[0]}, {key[1]}] already added.</p>"
                )
            else:
                state["spans"].append(sp)
                state["spans"].sort(key=lambda x: (x["word_start"], x["word_end"]))
                result_out.value = (
                    f"<p style='color:#0a7a2f'><b>Added</b> [{sp['word_start']}, {sp['word_end']}]</p>"
                    f"<pre style='white-space:pre-wrap;color:#000;background:#fff8e1;padding:8px;'>"
                    f"{html.escape(sp['phrase'])}</pre>"
                )
            state["pending_start"] = None
            _refresh_spans_html()
        _set_instruct()
        _refresh_words()

    if HAS_COLAB_CALLBACKS:
        colab_output.register_callback("span_word_click", on_word_index)

    def show_example():
        ex = current()
        state["spans"] = []
        state["pending_start"] = None
        n = max(0, ex["n_words"] - 1)
        start_box.max = n
        end_box.max = n
        start_box.value = 0
        end_box.value = min(4, n) if ex["n_words"] else 0

        header.value = (
            f"<h3 style='color:#000;'>Example {state['i'] + 1} / {len(queue)}</h3>"
            f"<p style='color:#000;'>row_id=<b>{ex['row_id']}</b> | persona_id=<b>{ex['persona_id']}</b> | "
            f"snippet words=<b>{ex['n_words']}</b></p>"
        )
        query_out.value = (
            "<p style='color:#000;'><b>Query</b></p>"
            f"<pre style='white-space:pre-wrap;background:#f7f7f7;padding:10px;color:#000;'>"
            f"{html.escape(ex['query'])}</pre>"
        )
        correct_out.value = (
            "<p style='color:#000;'><b>correct_answer</b></p>"
            f"<pre style='white-space:pre-wrap;background:#eef7ee;padding:10px;color:#000;'>"
            f"{html.escape(ex['correct_answer'])}</pre>"
        )
        pref_out.value = (
            "<p style='color:#000;'><b>preference</b></p>"
            f"<pre style='white-space:pre-wrap;background:#eef3ff;padding:10px;color:#000;'>"
            f"{html.escape(ex['preference'] or '(empty)')}</pre>"
        )
        snippet_plain.value = (
            "<p style='color:#000;'><b>Relevant snippet</b> "
            "(training/raw <code>str(snippet)</code>)</p>"
            f"<pre style='white-space:pre-wrap;background:#ffffff;border:1px solid #333;"
            f"padding:10px;color:#000000;'>{html.escape(ex['snippet'])}</pre>"
        )
        result_out.value = ""
        _set_instruct()
        _refresh_words()
        _refresh_spans_html()
        status.value = (
            f"<span style='color:#000;'>Saved this session: <b>{state['saved']}</b> | "
            f"Skipped: <b>{state['skipped']}</b> | "
            f"On disk: <b>{len(load_existing_labels())}</b></span>"
        )

    def on_add(_):
        try:
            sp = _make_span(current()["words"], int(start_box.value), int(end_box.value))
        except Exception as err:
            result_out.value = f"<p style='color:#b00020'>{html.escape(str(err))}</p>"
            return
        key = (sp["word_start"], sp["word_end"])
        if any((x["word_start"], x["word_end"]) == key for x in state["spans"]):
            result_out.value = f"<p style='color:#b00020'>Span already added.</p>"
            return
        state["spans"].append(sp)
        state["spans"].sort(key=lambda x: (x["word_start"], x["word_end"]))
        result_out.value = (
            f"<p style='color:#0a7a2f'><b>Added</b> [{sp['word_start']}, {sp['word_end']}]</p>"
            f"<pre style='white-space:pre-wrap;color:#000;background:#fff8e1;padding:8px;'>"
            f"{html.escape(sp['phrase'])}</pre>"
        )
        _refresh_spans_html()
        _refresh_words()

    def on_undo(_):
        if state["spans"]:
            removed = state["spans"].pop()
            result_out.value = (
                f"<p style='color:#000;'>Removed [{removed['word_start']}, {removed['word_end']}]</p>"
            )
        state["pending_start"] = None
        _set_instruct()
        _refresh_spans_html()
        _refresh_words()

    def on_clear(_):
        state["spans"] = []
        state["pending_start"] = None
        result_out.value = "<p style='color:#000;'>Cleared all phrases.</p>"
        _set_instruct()
        _refresh_spans_html()
        _refresh_words()

    def on_cancel_sel(_):
        state["pending_start"] = None
        result_out.value = "<p style='color:#000;'>Selection cancelled.</p>"
        _set_instruct()
        _refresh_words()

    def on_submit(_):
        ex = current()
        spans = list(state["spans"])
        if not spans:
            result_out.value = (
                "<p style='color:#b00020'><b>No phrases added.</b> "
                "Click start/end words (or use the index boxes).</p>"
            )
            return
        phrases = [sp["phrase"] for sp in spans]
        starts = [sp["word_start"] for sp in spans]
        ends = [sp["word_end"] for sp in spans]
        rec = {
            "subset": SUBSET_NAME,
            "split": "val",
            "row_id": ex["row_id"],
            "persona_id": ex["persona_id"],
            "query": ex["query"],
            "correct_answer": ex["correct_answer"],
            "preference": ex["preference"],
            "snippet": ex["snippet"],
            "snippet_format": "training_raw_str",
            "status": "labeled",
            "spans": spans,
            "n_spans": len(spans),
            "word_starts": starts,
            "word_ends": ends,
            "phrases": phrases,
            "word_start": starts[0],
            "word_end": ends[0],
            "phrase": phrases[0],
            "span_text": " | ".join(phrases),
            "n_span_words": sum(sp["n_words"] for sp in spans),
            "n_snippet_words": ex["n_words"],
            "tokenization": "str.split()",
            "annotator": ANNOTATOR_ID,
            "labeled_at": datetime.now(timezone.utc).isoformat(),
            "seed": SEED,
        }
        append_label(rec)
        existing[ex["row_id"]] = rec
        state["saved"] += 1
        result_out.value = (
            f"<p style='color:#0a7a2f'><b>Saved {len(spans)} phrase(s)</b></p>"
            f"<pre style='white-space:pre-wrap;background:#fff8e1;padding:10px;"
            f"border:1px solid #f0c36d;color:#000;'>{html.escape(rec['span_text'])}</pre>"
        )
        status.value = (
            f"<span style='color:#000;'>✔ Saved {len(spans)} span(s). "
            f"Session saved: <b>{state['saved']}</b>. Click <b>Next</b>.</span>"
        )

    def on_skip(_):
        ex = current()
        rec = {
            "subset": SUBSET_NAME,
            "split": "val",
            "row_id": ex["row_id"],
            "persona_id": ex["persona_id"],
            "query": ex["query"],
            "correct_answer": ex["correct_answer"],
            "preference": ex["preference"],
            "snippet": ex["snippet"],
            "snippet_format": "training_raw_str",
            "status": "skipped",
            "spans": [],
            "n_spans": 0,
            "word_starts": [],
            "word_ends": [],
            "phrases": [],
            "word_start": None,
            "word_end": None,
            "phrase": None,
            "span_text": None,
            "n_span_words": None,
            "n_snippet_words": ex["n_words"],
            "tokenization": "str.split()",
            "annotator": ANNOTATOR_ID,
            "labeled_at": datetime.now(timezone.utc).isoformat(),
            "seed": SEED,
        }
        append_label(rec)
        existing[ex["row_id"]] = rec
        state["skipped"] += 1
        if state["i"] < len(queue) - 1:
            state["i"] += 1
            show_example()
        else:
            show_example()
            status.value = f"<b>Done.</b> Labels file: {LABELS_PATH}"

    def on_back(_):
        if state["i"] > 0:
            state["i"] -= 1
            show_example()

    def on_next(_):
        if state["i"] < len(queue) - 1:
            state["i"] += 1
            show_example()

    btn_add.on_click(on_add)
    btn_undo.on_click(on_undo)
    btn_clear.on_click(on_clear)
    btn_cancel_sel.on_click(on_cancel_sel)
    btn_submit.on_click(on_submit)
    btn_skip.on_click(on_skip)
    btn_back.on_click(on_back)
    btn_next.on_click(on_next)

    ui = widgets.VBox([
        header,
        query_out,
        correct_out,
        pref_out,
        snippet_plain,
        widgets.HTML(
            "<p style='color:#000;'><b>Clickable snippet words</b> "
            "(black text; click start then end)</p>"
        ),
        instruct,
        words_html,
        spans_out,
        widgets.HTML(
            "<p style='color:#000;'><b>Fallback:</b> type indices if clicking is unavailable</p>"
        ),
        widgets.HBox([start_box, end_box, btn_add]),
        widgets.HBox([btn_undo, btn_clear, btn_cancel_sel]),
        widgets.HBox([btn_submit, btn_skip, btn_back, btn_next]),
        result_out,
        status,
    ])
    display(ui)
    show_example()
    print("Clickable HTML buttons:", "YES (Colab)" if HAS_COLAB_CALLBACKS else "NO (use index boxes)")

Clickable HTML buttons: YES (Colab)


In [6]:
from IPython.display import display

def load_labels_df(path=LABELS_PATH):
    rows = []
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
    return pd.DataFrame(rows)

labels_df = load_labels_df()
print("Labels file:", LABELS_PATH.resolve())
print("n records:", len(labels_df))
if len(labels_df):
    print(labels_df["status"].value_counts(dropna=False))
    cols = ["row_id", "persona_id", "status", "n_spans", "word_starts", "word_ends", "phrases", "span_text", "preference", "correct_answer", "query"]
    cols = [c for c in cols if c in labels_df.columns]
    display(labels_df[cols].tail(15))
    csv_path = ANNOT_DIR / f"{SUBSET_NAME}_val_relevant_spans_2.csv"
    labels_df.to_csv(csv_path, index=False)
    print("CSV:", csv_path)

Labels file: /content/drive/MyDrive/privacy_perserving_pllm/annotation/relevant_span_labels/all_subset_val_relevant_spans.jsonl
n records: 9
status
labeled    9
Name: count, dtype: int64


,row_id,persona_id,status,n_spans,word_starts,word_ends,phrases,span_text,preference,correct_answer,query
0,223,410,labeled,1,[69],[84],"[""Please forget my preference about worrying o...","""Please forget my preference about worrying ov...",Do not remember 'Worries about PTSD-like sympt...,You might carry a small bottle of water or a f...,What can I do to feel less anxious when I have...
1,558,449,labeled,2,"[100, 237]","[107, 246]",[small local eateries recommended by the schoo...,small local eateries recommended by the school...,Likes trying new restaurants,"Since you enjoy trying new restaurants, you mi...",I’ve got a free weekend coming up in the city—...
2,114,923,labeled,2,"[102, 210]","[105, 213]","[a quiet workspace downtown—I, a quiet downtow...",a quiet workspace downtown—I | a quiet downtow...,Uses public libraries regularly,Since you already make good use of public libr...,What are some budget-friendly ways to get acce...
3,300,637,labeled,1,[3],[22],"[""I’ve read that skiing has roots in military ...","""I’ve read that skiing has roots in military h...",Likes learning about historical events,Given your interest in historical events and a...,Can you suggest some really engaging podcasts ...
4,348,661,labeled,1,[50],[105],[two eager apprentices trailing behind her wit...,two eager apprentices trailing behind her with...,Feels a deep responsibility to mentor younger ...,If you want to support others without draining...,How can I support others without feeling compl...
5,95,861,labeled,3,"[79, 103, 417]","[80, 109, 422]","[metabolic disorders, longstanding health issu...",metabolic disorders | longstanding health issu...,Family history of type 2 diabetes on her mothe...,Since you have a family history of type 2 diab...,What are some healthier options to keep on han...
6,95,861,labeled,3,"[79, 103, 417]","[80, 109, 422]","[metabolic disorders, longstanding health issu...",metabolic disorders | longstanding health issu...,Family history of type 2 diabetes on her mothe...,Since you have a family history of type 2 diab...,What are some healthier options to keep on han...
7,95,861,labeled,3,"[79, 103, 417]","[80, 109, 422]","[metabolic disorders, longstanding health issu...",metabolic disorders | longstanding health issu...,Family history of type 2 diabetes on her mothe...,Since you have a family history of type 2 diab...,What are some healthier options to keep on han...
8,344,891,labeled,2,"[26, 109]","[39, 120]","[shaded bench in our local green space, watchi...","shaded bench in our local green space, watchin...",Enjoys quiet afternoons in parks,You could pack a light plant-based picnic and ...,What are some relaxing ways to spend an aftern...


CSV: /content/drive/MyDrive/privacy_perserving_pllm/annotation/relevant_span_labels/all_subset_val_relevant_spans_2.csv
